# 02 — Normalization analysis

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import HTML, display

from msflow import *
import msflow

pd.set_option("display.width", 180)

---
## 1 · Conservativity Analysis

We plot $\|\operatorname{curl}V\|^2/(\|\operatorname{curl}V\|^2+
\|\operatorname{div}V\|^2)$ on one box and grid.  Zero means conservative field.

In [ ]:
# Data, Grid, Kernel
DATASET, N, SEED = "ring", 320, 2026
SIGMA, GRID_N = 1.0, 36
KERNELS_TO_TEST = tuple(KERNELS)

# Euler
TAU, N_ITER, CLIP = 0.04, 600, None

# Metrics
PROBE_EVERY, STORE_EVERY = 25, 10
FLOOR_FRAC = 0.3
GAUGE_SEED, GAUGE_N, GAUGE_REPEATS = 7, 150, 6
GAUGE_ANCHORS, GAUGE_H, GAUGE_RIDGE = None, None, 1e-4
NORMALIZATION_MODES = ("none", "p", "q", "geo", "arith", "harm")
NORMALIZATION_SNAPSHOT_KS = (0, 100, 300, N_ITER)

# Batch Settings
BATCH_N_ITER, BATCH_EVAL_SIGMA, BATCH_STORE_EVERY = 400, 0.5, 20
BATCH_SNAPSHOT_KS, BATCH_PAGE_SIZE = (0, 80, 200, BATCH_N_ITER), 3
BATCH_N_ITER = 400
BATCH_EVAL_SIGMA = 0.5
BATCH_STORE_EVERY = 20
BATCH_SNAPSHOT_KS = (
    0,
    BATCH_N_ITER // 5,
    BATCH_N_ITER // 2,
    BATCH_N_ITER,
)
BATCH_PAGE_SIZE = 3

# Deng 
DENG_SELF_MASK = True
DENG_SELF_MASK_DISTANCE = 1e6

# Sinkhorn
SINKHORN_TOL = 1e-8
SINKHORN_MAX_ITER = 4000

data = load_dataset(DATASET, n=N, seed=SEED)
q_probe = data.q0

rows = []
for kname in KERNELS_TO_TEST:
    kern = get_kernel(kname, SIGMA)
    fields = {"row_equation": mean_shift(kern),
              "ms_unnorm": mean_shift_unnormalized(kern), "mmd": mmd(kern)}
    if kern.has_sharp:
        fields["sharp_core"] = sharp_core(kern)
    for fname, fld in fields.items():
        g, xx, yy, pts = msflow.viz.field_grid(data.extent, GRID_N)
        V = fld.at(pts, data.p, q_probe)
        rot = rotational_fraction(V[:, 0].reshape(xx.shape),
                                  V[:, 1].reshape(xx.shape), g)
        rows.append({"kernel": kname, "field": fname, "rot_fraction": rot})

rot_table = pd.DataFrame(rows).pivot(index="field", columns="kernel", values="rot_fraction")
save_table(rot_table, "02_rotational_fraction",
           caption="Finite-grid curl/divergence ratio; settings are part of the result.")
rot_table.round(4)

In [ ]:
kern_lap = get_kernel("Laplacian", SIGMA)
fig, rot = plot_field_gallery(
    {"population row field (Laplacian)": mean_shift(kern_lap),
     "sharp core G (Laplacian)": sharp_core(kern_lap),
     "population row field (Gaussian)": mean_shift(get_kernel("Gaussian", SIGMA))},
    data.p, q_probe, extent=data.extent, n_grid=GRID_N, figsize=(12, 9))
save(fig, "02_field_gallery");

## 2 · $PG + R$ decomposition


In [ ]:
base = mean_shift(kern_lap)
traj = run(data.q0, data.p, base, n_iter=N_ITER, tau=TAU, clip=CLIP,
           probes={"W2": lambda k, p, q, V: W2(q, p)}, probe_every=PROBE_EVERY,
           store_every=STORE_EVERY, label="population row (Laplacian)")

ks, parts = [], []
for i in range(0, len(traj.X), 5):
    q = traj.q_at_frame(i)
    d = decompose(q.X, data.p, q, kern_lap)
    l2 = lambda A: float(np.sqrt((q.w * (A ** 2).sum(-1)).sum()))
    ks.append(traj.iteration_of_frame(i))
    parts.append({"|P.G|": l2(d["P"][:, None] * d["G"]), "|R|": l2(d["R"]),
                  "|V|": l2(d["V"]), "mean a_q": float(d["a_q"].mean()),
                  "spread a_q": float(d["a_q"].std() / d["a_q"].mean())})
parts = pd.DataFrame(parts, index=ks)

fig, ax = plt.subplots(1, 3, figsize=(9.6, 3.2))
ax[0].semilogy(parts.index, parts["|V|"], lw=1.8, label=r"$\|V\|$")
ax[0].semilogy(parts.index, parts["|P.G|"], lw=1.6, ls="--", label=r"$\|PG\|$")
ax[0].semilogy(parts.index, parts["|R|"], lw=1.6, ls=":", label=r"$\|R\|$")
ax[0].set(xlabel="$k$", ylabel=r"$L^2(q^k)$ norm", title="magnitudes along the flow")
ax[0].legend(fontsize=8)
ax[1].plot(parts.index, parts["|R|"] / parts["|V|"], lw=1.8, color="#d62728")
ax[1].set(xlabel="$k$", ylabel=r"$\|R\|/\|V\|$", title="share of the residual")
ax[2].plot(parts.index, parts["mean a_q"], lw=1.8, label=r"mean $a_q$")
ax[2].plot(parts.index, parts["spread a_q"], lw=1.6, ls="--", label=r"rel. spread of $a_q$")
ax[2].axhline(SIGMA ** 2, color="0.4", ls=":", lw=1, label=r"$\sigma^2$ (Gaussian value)")
ax[2].set(xlabel="$k$", title="the preconditioner $P=a_q I$"); ax[2].legend(fontsize=8)
for a in ax: a.grid(alpha=.3)
fig.tight_layout(); save(fig, "02_pgr_along_flow");

In [ ]:
gauss_kern = get_kernel("Gaussian", SIGMA)
dec_g = decompose(data.q0.X, data.p, data.q0, gauss_kern)
print(f"Gaussian kernel:  max |R| = {np.abs(dec_g['R']).max():.2e}"
      f"   max |a_q - sigma^2| = {np.abs(dec_g['a_q'] - SIGMA**2).max():.2e}")

---
## 3 · How much is captured by a finite gradient basis?

In [ ]:
rng = np.random.default_rng(GAUGE_SEED)
regimes = {
    "far from p": lambda: Emp(
        data.q0.X[rng.choice(data.q0.n, GAUGE_N, replace=data.q0.n < GAUGE_N)]
        + 0.05 * rng.standard_normal((GAUGE_N, 2))),
    "near a mode": lambda: Emp(data.centers[0] + 0.5 * rng.standard_normal((GAUGE_N, 2))),
    "close to p": lambda: Emp(data.p.X[rng.choice(data.p.n, GAUGE_N)]
                              + 0.3 * rng.standard_normal((GAUGE_N, 2))),
}
rows = []
for name, sampler in regimes.items():
    gR, gG, nR, nG = [], [], [], []
    for _ in range(GAUGE_REPEATS):
        q = sampler()
        d = decompose(q.X, data.p, q, kern_lap)
        anchors = q.X if GAUGE_ANCHORS is None else np.asarray(GAUGE_ANCHORS)
        kwargs = dict(anchors=anchors, h=GAUGE_H, ridge=GAUGE_RIDGE)
        r_diag = gauge_fraction(q, d["R"], **kwargs)
        g_diag = gauge_fraction(q, d["G"], **kwargs)
        gR.append(r_diag["gauge_frac"]); gG.append(g_diag["gauge_frac"])
        nR.append(r_diag["norm"]); nG.append(g_diag["norm"])
    rows.append({"regime": name, "unexplained frac R": np.mean(gR),
                 "unexplained frac G": np.mean(gG), "|R|": np.mean(nR),
                 "|G|": np.mean(nG), "|R|/|G|": np.mean(nR) / np.mean(nG)})
gauge = pd.DataFrame(rows).set_index("regime")
save_table(gauge, "02_gauge", caption="Finite-basis gradient projection diagnostic.")
gauge.round(3)

---
## 4 · Common normalizations are less effective

In [ ]:
floor = density_floor(data.p, kern_lap, frac=FLOOR_FRAC)
print(f"density floor c = {floor:.3e}")

norm_runs = Runs()
norm_runs["sharp"] = run(
    data.q0, data.p, sharp_core(kern_lap), n_iter=N_ITER, tau=TAU, clip=CLIP,
    probes=standard_probes(data, kern_lap), probe_every=PROBE_EVERY,
    store_every=STORE_EVERY, label=r"sharp core $G$")
norm_runs["row_separate"] = run(
    data.q0, data.p, common_normalization(kern_lap, "row_separate"),
    n_iter=N_ITER, tau=TAU, clip=CLIP, probes=standard_probes(data, kern_lap),
    probe_every=PROBE_EVERY, store_every=STORE_EVERY, label="population row (separate)")
for mode in NORMALIZATION_MODES:
    mode_floor = 0.0 if mode == "none" else floor
    norm_runs[mode] = run(
        data.q0, data.p, common_normalization(kern_lap, mode, floor=mode_floor),
        n_iter=N_ITER, tau=TAU, clip=CLIP,
        probes=standard_probes(data, kern_lap), probe_every=PROBE_EVERY,
        store_every=STORE_EVERY, label=f"common {mode}")

fig = plot_runs(norm_runs, keys=("W2", "F", "n_covered", "beta_F"))
save(fig, "02_normalization_curves");

In [ ]:
norm_table = runs_table(norm_runs, columns=("W2", "F", "H", "n_covered", "entropy",
                                            "precision", "beta_F"),
                        n_modes=len(data.centers), tau=TAU)
save_table(
    norm_table, "02_normalization",
    caption=("Population row, sharp-core and common-denominator comparison "
             f"(ring, Laplacian sigma={SIGMA:g}, tau={TAU:g}, floor={floor:.3e})."))
style(norm_table)

In [ ]:
norm_names = list(norm_runs)
for page, start in enumerate(range(0, len(norm_names), 3), 1):
    names = norm_names[start:start + 3]
    page_runs = Runs({name: norm_runs[name] for name in names})
    fig = plot_snapshots(page_runs, data.p, ks=NORMALIZATION_SNAPSHOT_KS,
                         centers=data.centers, extent=data.extent)
    save(fig, f"02_normalization_snapshots_{page}")
    display(fig); plt.close(fig)

---
## 5 · Batch normalizations

We here compare the differents normalizations of interest.

- `deng_theoretical_row`
- `deng_official`
- `proxy_separate_laplace`
- `proxy_gaussian`
- `sinkhorn_Kinf`

The raw panel compares the fields exactly as implemented. The norm-matched
panel compares their directions after matching their $L^2(q)$ norms to the
theoretical Deng row field (to remove step size bias).

In [ ]:
proxy_gauss_kernel = get_kernel("Gaussian", SIGMA)

deng_theoretical = deng_population(kern_lap)

batch_fields = {
    "deng_theoretical_row": deng_theoretical,

    "deng_official": deng_official(
        temperature=SIGMA,
        self_mask=DENG_SELF_MASK,
        self_mask_distance=DENG_SELF_MASK_DISTANCE,
    ),

    "proxy_separate_laplace": sinkhorn_proxy(kern_lap),

    "proxy_gaussian": sinkhorn_proxy(proxy_gauss_kernel),

    "sinkhorn_Kinf": sinkhorn_drift(
        SIGMA,
        K=np.inf,
        tol=SINKHORN_TOL,
        max_iter=SINKHORN_MAX_ITER,
    ),
}


def run_batch_panel(candidates, match_norm):
    out = Runs()

    for name, candidate in candidates.items():
        is_reference = name == "deng_theoretical_row"

        if match_norm and not is_reference:
            fld = norm_matched(candidate, deng_theoretical)
            label = f"{name} (norm-matched)"
        else:
            fld = candidate
            label = name

        probes = standard_probes(
            data,
            field_kernel(fld),
            eval_sigma=BATCH_EVAL_SIGMA,
            r_cover=1.4,
            tau=TAU,
        )

        out[name] = run(
            data.q0,
            data.p,
            fld,
            n_iter=BATCH_N_ITER,
            tau=TAU,
            clip=CLIP,
            probes=probes,
            probe_every=PROBE_EVERY,
            store_every=BATCH_STORE_EVERY,
            label=label,
        )

    return out


batch_raw = run_batch_panel(batch_fields, match_norm=False)
batch_nm = run_batch_panel(batch_fields, match_norm=True)

fig = plot_runs(
    batch_raw,
    keys=("W2", "H", "n_covered", "Vnorm"),
    ncol=2,
    figsize=(8.5, 6.2),
)
fig.suptitle("Raw fields", y=1.01)
save(fig, "02_batch_normalizations_raw")

fig = plot_runs(
    batch_nm,
    keys=("W2", "H", "n_covered", "Vnorm"),
    ncol=2,
    figsize=(8.5, 6.2),
)
fig.suptitle("Norm-matched direction diagnostic", y=1.01)
save(fig, "02_batch_normalizations_norm_matched")

batch_table = pd.concat({
    "raw": runs_table(
        batch_raw,
        columns=("W2", "H", "n_covered", "Vnorm"),
    ),
    "norm_matched": runs_table(
        batch_nm,
        columns=("W2", "H", "n_covered", "Vnorm"),
    ),
})

save_table(
    batch_table,
    "02_batch_normalizations",
    caption=(
        f"Theoretical Deng row field, official Deng Algorithm 2, "
        f"Proxy and balanced Sinkhorn; T=sigma={SIGMA:g}, tau={TAU:g}."
    ),
)

batch_table.round(4)

In [ ]:
batch_names = list(batch_raw)

for page, start in enumerate(
    range(0, len(batch_names), BATCH_PAGE_SIZE),
    1,):
    names = batch_names[start:start + BATCH_PAGE_SIZE]
    page_runs = Runs({
        name: batch_raw[name]
        for name in names
    })

    fig = plot_snapshots(
        page_runs,
        data.p,
        ks=BATCH_SNAPSHOT_KS,
        centers=data.centers,
        extent=data.extent,
        panel_size=(1.8, 1.7),
    )

    save(fig, f"02_batch_normalizations_snapshots_{page}")
    display(fig)
    plt.close(fig)

---
## 6 · Coifman–Lafon $\alpha$

We numerically verify the stationnary density using Lafon exponent $\alpha$ is indeed $p^{2(1-\alpha)}$

In [ ]:
HIER_N, HIER_SEED, HIER_SIGMA = 480, 2026, 1.5
ALPHAS_OPERATOR = [0.0, 0.25, 0.5, 0.75, 1.0]
ALPHAS = [0.0, 0.25, 0.5, 0.75]
ALPHA_N_ITER, ALPHA_PROBE_EVERY, ALPHA_STORE_EVERY = 1500, 50, 50

hier = load_dataset("mass_hierarchy", n=HIER_N, seed=HIER_SEED)
print("target masses (heavy -> light):", np.round(hier.p_mass, 3))
kern_h = get_kernel("Gaussian", HIER_SIGMA)

spec = pd.DataFrame([
    {"alpha": a, **{k: v for k, v in alpha_operator(hier.p, kern_h, a).items()
                     if k in ("slope", "predicted_slope", "stationary_exponent")}}
    for a in ALPHAS_OPERATOR]).set_index("alpha")
save_table(spec, "02_alpha_operator",
           caption="Finite-sample operator exponent versus its asymptotic prediction.")
spec.round(3)

In [ ]:
ref = alpha_mean_shift(kern_h, 0.0)
alpha_runs, alpha_runs_nm = Runs(), Runs()
for a in ALPHAS:
    fld = alpha_mean_shift(kern_h, a)
    common = dict(n_iter=ALPHA_N_ITER, tau=TAU, clip=CLIP, probes=standard_probes(hier, kern_h),
                  probe_every=ALPHA_PROBE_EVERY, store_every=ALPHA_STORE_EVERY)
    alpha_runs[f"a{a}"] = run(hier.q0, hier.p, fld, label=rf"$\alpha={a}$", **common)
    alpha_runs_nm[f"a{a}nm"] = run(hier.q0, hier.p, norm_matched(fld, ref),
                                   label=rf"$\alpha={a}$ (nm)", **common)

fig = plot_runs(alpha_runs, keys=("W2", "mass_err", "H", "entropy"))
fig.suptitle("raw", y=1.01); save(fig, "02_alpha_raw")
fig = plot_runs(alpha_runs_nm, keys=("W2", "mass_err", "H", "entropy"))
fig.suptitle("norm-matched to $\\alpha=0$", y=1.01); save(fig, "02_alpha_nm");

In [ ]:
def light_mode_mass(r):
    m = mode_metrics(r.q_final, hier.centers)["mass"]
    return {"light mode mass": float(m[-1]), "target": float(hier.p_mass[-1])}

alpha_table = pd.concat([
    runs_table(alpha_runs, columns=("W2", "mass_err", "H", "entropy"), extra=light_mode_mass),
    runs_table(alpha_runs_nm, columns=("W2", "mass_err", "H", "entropy"), extra=light_mode_mass),
])
save_table(alpha_table, "02_alpha",
           caption=r"Coifman--Lafon $\alpha$ on a geometric mass hierarchy, raw and "
                   r"norm-matched to $\alpha=0$.")
alpha_table.round(4)

In [ ]:
alpha_snapshot_runs = Runs({**alpha_runs, **alpha_runs_nm})
alpha_names = list(alpha_snapshot_runs)
for page, start in enumerate(range(0, len(alpha_names), 3), 1):
    names = alpha_names[start:start + 3]
    page_runs = Runs({name: alpha_snapshot_runs[name] for name in names})
    fig = plot_snapshots(page_runs, hier.p,
                         ks=[0, 200, 700, ALPHA_N_ITER],
                         centers=hier.centers, extent=hier.extent)
    save(fig, f"02_alpha_snapshots_{page}")
    display(fig); plt.close(fig)